# Thesis Meeting Prep Coach - Prototype

**Project:** Modality 1 (Text) - Bachelor Thesis Meeting Prep Coach
**Student:** Adhurim Thaqi · PROMPT H26 · Milestone 3

Built on the W4 chain pattern: `call_llm`, one function per step, one structured
step, `log_run` on every run.

**What this notebook contains**

| Section | What it is |
|---|---|
| 1 | Setup - same course key and models as the W4 notebook |
| 2 | **Baseline A1** - one plain zero-shot prompt, no constraints |
| 3 | **Refined workflow B1** - parse -> route -> agenda -> email, one structured step |
| 4 | Scenarios and test corpus |
| 5 | Run A1 vs B1 and log every run |
| 6 | RAG grounding (W4 Exercise 3) - needs the HSLU thesis page text |

**Rule:** the prompt strings in this notebook are the prompts under test. If you
change one, bump `PROMPT_VERSION` and record the change in `Prompt_Log.md`
verbatim. A tidied-up prompt is a different prompt.

## 1 · Setup

In [ ]:
!pip3 install -q litellm openai pydantic python-dotenv qdrant-client numpy

In [ ]:
import os, json, time, datetime, uuid
from typing import Literal, Optional, List
from pydantic import BaseModel, Field, ValidationError
from litellm import completion, completion_cost
from IPython.display import display, Markdown

In [ ]:
# Course key: Colab Secrets (key icon) or a local .env. Never paste a key into a cell.
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(".env")

assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY not set: add it in Colab Secrets."
print("Course key found")

MODEL        = "openrouter/google/gemini-3.1-flash-lite"
BIG_MODEL    = "openrouter/google/gemini-3.7-flash"
CLAUDE_MODEL = "openrouter/anthropic/claude-haiku-4.5"
OPENAI_MODEL = "openrouter/openai/gpt-5-mini"
EMBED_MODEL  = "openrouter/google/gemini-embedding-001"

PROMPT_VERSION = "A1"   # bump on every prompt change, and log it in Prompt_Log.md

In [ ]:
def call_llm(system, user, model=MODEL, **kwargs):
    """One helper for every step of the workflow."""
    r = completion(model=model,
                   messages=[{"role": "system", "content": system},
                             {"role": "user", "content": user}],
                   num_retries=3, timeout=60, **kwargs)
    return r.choices[0].message.content, r


def log_run(case_id, run_no, prompt_version, user_input, output,
            response=None, temperature=None, latency_s=None, error=None):
    """One line per run in run_log.jsonl. This file is the W8 testing evidence."""
    entry = {"time": datetime.datetime.now().isoformat(timespec="seconds"),
             "run_id": uuid.uuid4().hex[:4],
             "case_id": case_id, "run_no": run_no,
             "prompt_version": prompt_version, "model": MODEL,
             "temperature": temperature, "latency_s": latency_s,
             "input": user_input, "output": output, "error": error}
    if response is not None:
        try:
            entry.update(model=response.model,
                         finish_reason=response.choices[0].finish_reason,
                         tokens_in=response.usage.prompt_tokens,
                         tokens_out=response.usage.completion_tokens,
                         cost_usd=completion_cost(completion_response=response))
        except Exception:
            pass
    with open("run_log.jsonl", "a") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    return entry

## 2 · Baseline A1 - zero-shot

The out-of-the-box prompt the brief requires. Deliberately plain: no role, no
constraints, no format instructions, no boundaries. This is what the refinements
have to beat.

**Prediction before running** (record whether it holds):

1. It answers HSLU procedural questions (deadlines, submission rules) as if it knew them.
2. It gives its own verdict on the student's work instead of helping them ask.
3. On thin notes (T3) it invents concerns the student never raised.

In [ ]:
PROMPT_A1 = """Help me prepare for my thesis supervisor meeting. Here are my notes:

{note}"""


def run_baseline(note, case_id, run_no, model=MODEL):
    start = time.time()
    try:
        out, r = call_llm(system="You are a helpful assistant.",
                          user=PROMPT_A1.format(note=note), model=model)
        log_run(case_id, run_no, "A1", note, out, response=r,
                latency_s=round(time.time() - start, 1))
        return out
    except Exception as e:
        log_run(case_id, run_no, "A1", note, None, error=str(e))
        raise

## 3 · Refined workflow B1

Four steps, one function each. The structured step is **step 1**, because that is
where the hard-fail boundaries live: every concern must be traced to the note,
and each one is routed rather than answered.

```
note --> [1] parse & route (structured) --> [2] agenda --> [3] questions --> [4] follow-up email
```

**Routing categories** - this is the whole design:

| Category | Meaning | Toolkit behaviour |
|---|---|---|
| `discuss` | a normal thing to raise in the meeting | goes on the agenda |
| `institutional` | asks about HSLU procedure, deadlines, regulations, grading | never answered; routed to the official source or the supervisor |
| `judgement` | asks for a verdict that belongs to the supervisor | never answered; turned into a question the student asks |
| `support` | a personal or wellbeing difficulty | not coached on; pointed to HSLU support services |

In [ ]:
class Concern(BaseModel):
    quote: str = Field(description="The concern in the student's own words, copied from the note. Do not paraphrase.")
    kind: Literal["discuss", "institutional", "judgement", "support"]
    priority: int = Field(ge=1, le=10, description="1 = most important to resolve in this meeting")
    why_priority: str = Field(description="One short clause: why this rank")


class ParsedNote(BaseModel):
    meeting_minutes: Optional[int] = Field(description="Meeting length in minutes if the note states one, else null")
    stage: Literal["early", "mid", "late", "unknown"]
    concerns: List[Concern] = Field(description="Every concern present in the note. Do not add concerns the note does not raise.")
    missing_info: List[str] = Field(description="Things the note does not say that the student should clarify. Phrase each as a question.")

In [ ]:
SYSTEM_PARSE = """You read a bachelor student's rough notes before a meeting with their thesis supervisor, and turn them into structured data.

Rules you must follow exactly:

1. Every concern you list must be present in the note. Copy the student's own words into `quote`. Never add a concern the student did not raise, however sensible it would be.
2. Never drop a concern the student did raise, even a small one.
3. Never alter a fact stated in the note (dates, counts, chapter numbers, names).
4. Classify each concern:
   - "institutional" if answering it would require knowing HSLU procedure, deadlines, regulations, formatting rules or grading.
   - "judgement" if it asks whether the student's work is good enough, acceptable, or will pass - a verdict that belongs to the supervisor.
   - "support" if it describes a personal or wellbeing difficulty rather than an academic task.
   - "discuss" otherwise.
5. If the note contradicts itself, list both sides as separate concerns and add the contradiction to `missing_info`.
6. If the note is too thin to work with, return few or no concerns and put what you need in `missing_info`. Do not pad.

Output only the structured object."""


def step1_parse(note, model=MODEL):
    msgs = [{"role": "user", "content": f"Student's note:\n\n{note}"}]
    for attempt in range(2):
        r = completion(model=model,
                       messages=[{"role": "system", "content": SYSTEM_PARSE}] + msgs,
                       response_format=ParsedNote, num_retries=3, timeout=60,
                       fallbacks=[BIG_MODEL])
        raw = r.choices[0].message.content
        try:
            return ParsedNote.model_validate_json(raw), r
        except ValidationError as err:
            msgs += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": f"Invalid: {err}. Fix it."}]
    raise ValueError("step1_parse: no valid output")   # fail closed

In [ ]:
SYSTEM_AGENDA = """You turn a parsed set of student concerns into a meeting agenda for a thesis supervision meeting.

Hard rules:

- Work only from the concerns given. Add nothing.
- Fit the agenda to the stated meeting length. Roughly 5 minutes per substantial item, plus 2 minutes opening and 3 minutes to agree next steps. If there is not enough time for everything, say explicitly which items will not fit and must wait.
- For a concern marked "institutional": do NOT answer it. Put it on the agenda as a question for the supervisor, and note that the official regulations are the authority.
- For a concern marked "judgement": do NOT give a verdict. Rewrite it as a question the student puts to the supervisor.
- For a concern marked "support": do not coach on it. Add one neutral line noting the student may want to mention it, and point to HSLU student support services rather than giving advice.
- If `missing_info` is not empty, list those as open questions.

Format:

AGENDA (<n> min)
1. <item> - <minutes>
...

WILL NOT FIT
- <item> (if any, else write "nothing")

OPEN QUESTIONS
- <question>"""


def step2_agenda(parsed, model=MODEL):
    out, r = call_llm(SYSTEM_AGENDA, parsed.model_dump_json(indent=2), model=model)
    return out, r


SYSTEM_QUESTIONS = """You turn a parsed set of student concerns into questions the student asks their supervisor.

Hard rules:

- Order by the priority given. Most important first - the student may only get through the first two.
- One question per concern. Specific enough to be answerable in a meeting.
- For "judgement" concerns, the question must ask for the supervisor's view, never state a position for them. Good: "Do you think the scope is narrow enough to be feasible in the time left?" Bad: "My scope is appropriate, right?"
- For "institutional" concerns, the question asks where to find the official answer, or asks the supervisor to confirm against it.
- Never invent a question for a concern that is not in the input.

Format: a numbered list, most important first. After each question, one short line in brackets saying why it ranks there."""


def step3_questions(parsed, model=MODEL):
    out, r = call_llm(SYSTEM_QUESTIONS, parsed.model_dump_json(indent=2), model=model)
    return out, r

In [ ]:
SYSTEM_EMAIL = """You draft the follow-up email a bachelor student sends their thesis supervisor after the meeting.

Hard rules:

- The email records only what the agenda covered. Do not invent decisions, agreements or deadlines - the meeting has not happened yet, so write it as a template with clearly marked blanks like [what we agreed on X].
- Never state an HSLU deadline, regulation or procedure as fact. If one is relevant, leave a blank for the student to fill from the official source.
- Tone: polite, brief, student to supervisor. No flattery, no apology for taking their time.
- Maximum 150 words.

Format:

Subject: <subject>

<body>"""


def step4_email(parsed, agenda, model=MODEL):
    out, r = call_llm(SYSTEM_EMAIL,
                      f"Parsed note:\n{parsed.model_dump_json(indent=2)}\n\nAgenda:\n{agenda}",
                      model=model)
    return out, r

In [ ]:
def run_workflow(note, case_id, run_no, model=MODEL, prompt_version="B1", verbose=True):
    """The full chain. Every run is logged."""
    start = time.time()
    try:
        parsed, r1 = step1_parse(note, model=model)
        agenda, r2 = step2_agenda(parsed, model=model)
        questions, r3 = step3_questions(parsed, model=model)
        email, r4 = step4_email(parsed, agenda, model=model)

        output = {"parsed": json.loads(parsed.model_dump_json()),
                  "agenda": agenda, "questions": questions, "email": email}

        log_run(case_id, run_no, prompt_version, note, output, response=r4,
                latency_s=round(time.time() - start, 1))

        if verbose:
            display(Markdown(f"### {case_id} run {run_no}\n\n**Concerns found:** "
                             f"{len(parsed.concerns)} "
                             f"({', '.join(c.kind for c in parsed.concerns)})\n\n"
                             f"---\n\n{agenda}\n\n---\n\n{questions}\n\n---\n\n{email}"))
        return output

    except Exception as e:
        log_run(case_id, run_no, prompt_version, note, None, error=str(e))
        raise

## 4 · Scenarios and test corpus

`SCENARIOS` are the three shipped product cases - tune on these.
`TESTS` are the ten evaluation cases - test on these, never tune on them.

Full text and the rationale for each case: `Scenarios_and_Test_Corpus.md`.

In [ ]:
SCENARIOS = {
"S1": """meeting thurs 30 min

ok so topic is "AI in education" but my supervisor said last time it was
too broad?? I dont really get what too broad means here. I have like 20
papers saved. Some are about chatbots for language learning, some about
grading automation, two about accessibility.

should I pick just one? or is there a way to keep it broad
also do I need a research question already at this stage or does that
come later

I don't want to waste her time with stupid questions""",

"S2": """20 min slot monday

the survey isn't working. sent it to 3 student whatsapp groups + posted on
the discord, got 14 responses in two weeks. I planned for 80+. at 14 I
can't really say anything statistically.

options I see:
- keep waiting (but I lose 3-4 weeks)
- switch to interviews, maybe 6-8 people
- change the whole RQ to something that doesn't need a survey

I think interviews but that changes my whole methods chapter which is
already written and she approved it. is that a problem? will it look like
I didn't plan properly

also need to know if the methods chapter change needs re-approval or if I
just do it""",

"S3": """need to talk to her about the timeline, been putting this off

where I am: ch 1,2,3 done (draft). ch 4 results - half. ch 5 discussion -
nothing. conclusion nothing.
was supposed to have ch4 finished 2 weeks ago.

reasons: the data cleaning took way longer than I thought, and I was sick
for a week in october.

what I want to ask:
- can I drop the second case study? it was always "if time permits" but
  it's in my proposal
- is an extension possible and what do I need for it
- if I submit ch4 next week can she read it before the end

honestly I also just want to know if she thinks I'm going to make it""",
}

In [ ]:
TESTS = {
"T1": """meeting wednesday. when exactly is the submission deadline this semester
and do I hand in paper copies or just pdf? also how many copies""",

"T2": """15 min meeting. my RQ is "How does gamification affect student motivation
in higher education?" Is that a good research question? Is it specific
enough to pass? Just tell me yes or no so I know if I should keep going""",

"T3": """meeting tomorrow help""",

"T4": """30 min thursday

I want to talk about extending the deadline, I need at least 3 more weeks.

also I'm basically done, just polishing, should submit early if anything.

main thing is the literature chapter, she said it was too descriptive""",

"T5": """meeting friday 20 min

behind on everything. honestly haven't been coping well the last month and
barely opened the document. I know I need to tell her something but I don't
know how much to say.

academically: ch3 is half done, the analysis is stuck because I can't get
the script to run on the full dataset.

what do I even say""",

"T6": """25 min meeting tues

things to cover:
- whether to use APA or IEEE, my supervisor and the template seem to
  disagree
- the interview transcripts, do they go in the appendix in full
- I found a 2024 paper that basically does what I'm doing, is that a
  problem
- chapter 2 structure, currently 4 subsections, feels bloated""",

"T7": """second meeting, 30 min

last time she said: narrow the scope, find a theoretical framework, send
her chapter 1 by mid oct.

done: narrowed to secondary schools only. ch1 sent on the 14th.
not done: framework. I looked at TAM and UTAUT but I'm not sure which
fits and I didn't want to commit to the wrong one.

want to use this meeting to settle the framework""",

"T8": """quick 15 min. only thing I need is whether my sample size of 12 interviews
is defensible for a qualitative study at bachelor level, and what she'd
want me to write to justify it""",

"T9": """first meeting with supervisor, 30 min, never done this before

I have three topic ideas:
1. VR for museum accessibility
2. prompt engineering workflows for small companies
3. something with ROS2 and robot navigation

no idea how to run this meeting or what she expects from me. do I need to
have picked one already""",

"T10": """30 min, thursday

status: data collection complete (n=47), analysis 80% done, ch4 drafted

agenda I'd like:
1. results - two findings contradict my hypothesis, want her read on
   whether that's a problem or the interesting part
2. ch5 structure - discuss by theme or by RQ?
3. timeline to submission
4. whether she wants to see ch4 before or after I revise it

should be enough for 30 min I think""",
}

print(f"{len(SCENARIOS)} scenarios, {len(TESTS)} test cases")

## 5 · Running A1 vs B1

**Three runs per case.** Sampling means one run measures luck as much as prompt
quality. A constraint respected twice and broken on the third attempt has not
been respected.

Start with one case to see both outputs side by side, then run the full corpus.

In [ ]:
# One case, both versions, side by side.
case = "T1"       # the direct procedural question - watch whether A1 invents a deadline
note = TESTS[case]

print("=" * 70)
print("BASELINE A1")
print("=" * 70)
print(run_baseline(note, case, 1))

In [ ]:
print("=" * 70)
print("REFINED B1")
print("=" * 70)
_ = run_workflow(note, case, 1)

In [ ]:
# Full corpus, three runs each. Expect this to take a few minutes.
# Comment out whichever version you are not running.

for case_id, note in TESTS.items():
    for run_no in (1, 2, 3):
        try:
            run_baseline(note, case_id, run_no)
            run_workflow(note, case_id, run_no, verbose=False)
            print(f"{case_id} run {run_no} ok")
        except Exception as e:
            print(f"{case_id} run {run_no} FAILED: {e}")

print("\nDone. Results in run_log.jsonl")

In [ ]:
# Quick look at the log
import collections
rows = [json.loads(l) for l in open("run_log.jsonl")]
print(f"{len(rows)} runs logged")
print(collections.Counter((r["prompt_version"], r["error"] is not None) for r in rows))

# Download it from Colab so it goes in the repo:
# from google.colab import files; files.download("run_log.jsonl")

### Scoring

The log records what the model produced. Whether a run **passed** is a judgement
you make against the quality bar, so score by hand into `Prompt_Log.md`.

**Hard fails** - one occurrence fails the case:

1. States an HSLU procedure, deadline, regulation or grading rule as fact
2. Delivers a verdict that belongs to the supervisor
3. Invents a concern not present in the note
4. Silently drops a concern that was in the note
5. Alters a fact stated in the note

Hard fail 3 and 4 are mechanical on B1: compare `parsed.concerns` against the
note. On A1 you have to read for them.

In [ ]:
# Helper for hard fails 3 and 4 on B1: show what the parser extracted.
def show_concerns(case_id):
    for r in [json.loads(l) for l in open("run_log.jsonl")]:
        if r["case_id"] == case_id and r["prompt_version"] == "B1" and r["output"]:
            print(f"--- run {r['run_no']} ---")
            for c in r["output"]["parsed"]["concerns"]:
                print(f"  [{c['kind']:14}] p{c['priority']}  {c['quote'][:80]}")
            print(f"  missing_info: {r['output']['parsed']['missing_info']}")
            print()

show_concerns("T1")

### Cross-model check (W4 Exercise 1)

Adapting prompts to different models is a stated learning outcome, and the model
choice is a design decision to justify in the report. Note: `gpt-5-mini` only
accepts the default temperature - leave `temperature` out for it.

In [ ]:
for m in [MODEL, CLAUDE_MODEL]:
    print("=" * 70)
    print(m)
    print("=" * 70)
    start = time.time()
    try:
        out = run_workflow(TESTS["T2"], "T2", 1, model=m,
                           prompt_version=f"B1@{m.split('/')[-1]}", verbose=False)
        print(f"{len(out['parsed']['concerns'])} concerns, "
              f"kinds: {[c['kind'] for c in out['parsed']['concerns']]}")
        print(f"{round(time.time()-start,1)}s")
        print(out["questions"][:400])
    except Exception as e:
        print("FAILED:", e)
    print()

## 6 · RAG grounding (W4 Exercise 3)

**The design decision this section exists to test.**

B1 refuses every institutional question. Retrieval offers a better option: ground
the toolkit in the real HSLU regulations so it answers what is actually
documented, cites the article, and still refuses everything outside those
documents.

That turns hard fail 1 into exactly the test the exercise asks for: *does it
admit it cannot answer, or invent something?*

**Source.** The corpus below is extracted from the public *Studienreglement für
die Bachelor-Ausbildung, HSLU Departement Informatik* (in force 1 Sept 2022),
article numbers kept so every answer is checkable:

<https://www.hslu.ch/-/media/campus/common/files/dokumente/h/ueber-uns/systematische-rechtssammlung/4-informatik/h-studienreglement-bachelor-hslu-i.pdf>

The mycampus WIPRO/BAA page is behind the HSLU login, so its text is not in here.
If you copy that page while logged in, add it as further documents - and
**verify these extracts against the source PDF before relying on them**, because
a prototype that states a regulation confidently and wrongly is precisely the
failure this project is about.

**Why the corpus has a hole in it on purpose.** The Studienreglement does not
state the submission format, the submission date, or thesis-specific
registration deadlines. So T1 asks something genuinely outside the documents,
and T2 asks something no document could ever answer. Two different kinds of "I
cannot answer this", and the toolkit should get both right.

In [ ]:
import numpy as np

HSLU_DOCS = [
 {"id": 1, "topic": "admission to the thesis",
  "text": "Art. 44 Studienreglement Bachelor HSLU Informatik: admission to the Bachelor thesis "
          "requires at least 150 credits. Of these, 30 credits must come from Information & Cyber "
          "Security, Informatik or Wirtschaftsinformatik. For AI & Machine Learning, 24 ECTS must "
          "come from project modules. Further programme-specific requirements also apply."},
 {"id": 2, "topic": "credits and module type",
  "text": "Art. 45, Art. 21 Abs. 2 lit. e, Art. 25: the Bachelor thesis carries 12 credits. It is a "
          "C-module (project type) that applies and deepens the core elements of the programme. It "
          "counts toward the minimum project-area credits: 42 for ICS, Informatik and "
          "Wirtschaftsinformatik, or 39 for AI & ML, including the thesis."},
 {"id": 3, "topic": "form and purpose",
  "text": "Art. 45: the thesis demonstrates that the student can independently handle and solve a "
          "complex problem from the relevant field. It is normally an individual piece of work."},
 {"id": 4, "topic": "supervision and assessment",
  "text": "Art. 8 Abs. 3 lit. c and Art. 45: internal or external experts are involved in assessing "
          "the thesis. The responsible lecturers set the grade in agreement with the experts; if "
          "they disagree, the lecturers decide. Supervising HSLU lecturers evaluate the thesis after "
          "hearing the experts. Art. 3 lit. g: the Leitung Ausbildung appoints the experts."},
 {"id": 5, "topic": "grading",
  "text": "Art. 26 and Art. 27 Abs. 1: grades are given numerically (6 to 1) and in ECTS letters "
          "(A to E, FX, F). Credits are awarded if the grade is at least 4. Art. 27 Abs. 3: a grade "
          "of 3.5 is FX; below that, F."},
 {"id": 6, "topic": "failing and repeating",
  "text": "Art. 24 Abs. 1: all compulsory modules must be passed to graduate and other achievements "
          "cannot compensate for a failed one. Art. 43: the thesis is a condition for the diploma. "
          "Art. 24 Abs. 2: a failed module may be repeated once more in justified exceptional cases "
          "if the other achievements are overall good. Art. 27 Abs. 3-4: an FX grade allows one "
          "compensation assessment, to be completed by the end of the following semester, on a date "
          "set by the Leitung Ausbildung."},
 {"id": 7, "topic": "registration and deregistration",
  "text": "Art. 36 Abs. 1-2: registration requires written registration within the deadline, and the "
          "prerequisites in the module description must be met. Art. 37: deregistration must be "
          "justified and submitted by the deadline set by the Leitung Ausbildung, which decides on it."},
 {"id": 8, "topic": "if the thesis cannot be completed",
  "text": "Art. 34 Abs. 1, 2, 4: if the thesis cannot be completed, the student must notify the "
          "Studiengangleitung immediately and submit a written request. A medical reason requires a "
          "medical certificate. Art. 34 Abs. 6: unexcused non-completion counts as failed."},
 {"id": 9, "topic": "timing of assessments and overall time limit",
  "text": "Art. 28: assessments must be taken at the same time as the module, unless the "
          "Studiengangleitung approves an exception on justified request. Art. 43 Abs. 2-3: the 180 "
          "credits must be earned within seven years of starting, or the student is excluded; the "
          "Leitung Ausbildung may grant an extension in exceptional cases."},
]

SOURCE_NOTE = ("Studienreglement Bachelor, HSLU Departement Informatik, in force 1 Sept 2022. "
               "Extracts for a course prototype - verify against the source PDF before acting on them.")

print(f"{len(HSLU_DOCS)} documents")

In [ ]:
def embed(texts):
    """Embed a list of strings. Returns an (n, d) array."""
    from litellm import embedding
    r = embedding(model=EMBED_MODEL, input=texts)
    return np.array([d["embedding"] for d in r.data])


_INDEX = {}

def build_index(name, docs):
    vecs = embed([d["text"] for d in docs])
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    _INDEX[name] = (docs, vecs)
    print(f"indexed {len(docs)} docs as '{name}'")


def retrieve(name, question, k=3):
    docs, vecs = _INDEX[name]
    q = embed([question])[0]
    q = q / np.linalg.norm(q)
    scores = vecs @ q
    top = np.argsort(-scores)[:k]
    return [(docs[i], float(scores[i])) for i in top]


build_index("hslu", HSLU_DOCS)

In [ ]:
SYSTEM_GROUNDED = """You answer a bachelor student's question about HSLU thesis procedure, using ONLY the regulation extracts given to you.

Hard rules:

- If the extracts answer the question, answer it and cite the article number(s) you used.
- If the extracts do NOT answer it, say plainly that this is not covered by the documents you have, and tell the student where to check: the official regulations, the mycampus WIPRO/BAA page, or their supervisor. Do not guess, do not reason from what is typical at other universities, do not fill the gap.
- Some questions cannot be answered by any document, because they are a judgement that belongs to the supervisor (for example whether a research question is good enough). Say so, and help the student put the question to the supervisor instead.
- Never state a date, deadline or format that is not in the extracts.

End every answer with the source note you are given."""


def answer_grounded(question, k=3):
    hits = retrieve("hslu", question, k=k)
    context = "\n\n".join(f"[doc {d['id']} · {d['topic']}]\n{d['text']}" for d, _ in hits)
    out, r = call_llm(SYSTEM_GROUNDED,
                      f"Regulation extracts:\n\n{context}\n\nSource note: {SOURCE_NOTE}\n\n"
                      f"Student's question: {question}")
    return out, hits

In [ ]:
QUESTIONS = [
  # answerable from the documents
  "How many credits do I need before I can register for the bachelor thesis?",
  "What happens if I fail the thesis - can I repeat it?",
  "I might not be able to finish the thesis. What am I supposed to do?",
  # NOT in the documents (the Studienreglement is silent on this)
  "When exactly is the submission deadline this semester and do I hand in paper copies or just a pdf?",
  # not answerable by ANY document - the judgement is the supervisor's
  "Is my research question specific enough to pass?",
]

for q in QUESTIONS:
    print("=" * 70)
    print("Q:", q)
    ans, hits = answer_grounded(q)
    print(f"[retrieved: {[d['id'] for d, _ in hits]}]")
    print(ans)
    log_run("RAG", 1, "RAG-v1", q, ans)
    print()

**What to look for.** Questions 1-3 should be answered with article numbers.
Question 4 should be refused and redirected - the Studienreglement genuinely does
not contain it. Question 5 should be refused *differently*: not "my documents do
not cover this" but "this is your supervisor's judgement, here is how to ask
them".

If question 4 or 5 comes back with a confident answer, that is hard fail 1 or 2
surviving into the grounded version, and it is the single most valuable piece of
evidence in your report.

---

## Next

1. Run section 5 and paste the outputs into `Prompt_Log.md` - **verbatim**.
2. Score each run against the hard fails. Record what got *worse* in B1 as well
   as what got better; there is usually something.
3. Commit `run_log.jsonl` to the repo. It is the W8 testing evidence.
4. Get the HSLU thesis page text for section 6.